# CIS 5190 Final Project — Track B: Fox vs NBC Headline Classification
## Live Demo Notebook

**Team 36:** Allison Mi, Amish Sethi, Kerry Zhao  
**Final leaderboard score:** **85.75%** (sub_144) — `+19.26 pp` over baseline, `+5.33 pp` over the next visible team

This notebook walks through the three results that drove our score:

1. **Distribution-matched data beats more data** (7.5k curated > 48k full scrape)
2. **Architectural diversity beats seed diversity** (DistilBERT + ALBERT + MPNet + ConvBERT + stylo)
3. **Ensemble accuracy saturates** (the 6th transformer is a regression, not an improvement)

All data loaded live from our public HuggingFace dataset: [`ASethi04/cis5190-fox-nbc-headlines`](https://huggingface.co/datasets/ASethi04/cis5190-fox-nbc-headlines).

GitHub: [`alliemi/cis5190-finalproject`](https://github.com/alliemi/cis5190-finalproject)

In [1]:
# Setup — runs in ~30s on Colab
%pip install -q datasets scikit-learn matplotlib seaborn pandas numpy

/home/asethi04/cis5190-finalproject/.venv/bin/python3: No module named pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset

plt.rcParams['figure.dpi'] = 110
plt.rcParams['font.size'] = 11
sns.set_style('whitegrid')

## 1. Load our public dataset from Hugging Face

We scraped both outlets' sitemaps and released the full 52k corpus + five canonical splits.

In [3]:
REPO = 'ASethi04/cis5190-fox-nbc-headlines'
all_clean = load_dataset(REPO, 'all_clean')['train'].to_pandas()
co_val_temp = load_dataset(REPO, 'co_val_temp')['train'].to_pandas()
val_frozen = load_dataset(REPO, 'val_frozen')['train'].to_pandas()
temp_val = load_dataset(REPO, 'temporal_val_frozen')['train'].to_pandas()
course_only = load_dataset(REPO, 'course_only')['train'].to_pandas()

splits = pd.DataFrame([
    ('all_clean (full corpus)', len(all_clean), (all_clean['label']==1).sum(), (all_clean['label']==0).sum()),
    ('course_only (starter)', len(course_only), (course_only['label']==1).sum(), (course_only['label']==0).sum()),
    ('co_val_temp (final training)', len(co_val_temp), (co_val_temp['label']==1).sum(), (co_val_temp['label']==0).sum()),
    ('val_frozen (random val)', len(val_frozen), (val_frozen['label']==1).sum(), (val_frozen['label']==0).sum()),
    ('temporal_val_frozen (test proxy)', len(temp_val), (temp_val['label']==1).sum(), (temp_val['label']==0).sum()),
], columns=['Split', 'Rows', 'Fox (1)', 'NBC (0)'])
splits

,Split,Rows,Fox (1),NBC (0)
0,all_clean (full corpus),52530,17657,34873
1,course_only (starter),3801,2000,1801
2,co_val_temp (final training),7532,3701,3831
3,val_frozen (random val),1967,911,1056
4,temporal_val_frozen (test proxy),1764,790,974


## 2. The task is harder than it looks

Fox and NBC cover the *same* news events — Trump, Israel, Harris, Gaza. The signal is in **how** they write, not **what** they write.

In [6]:
print('=== Same-day headlines on overlapping topics ===\n')
print('FOX  ->', all_clean[all_clean['label']==1]['headline'].sample(8, random_state=7).tolist())
print()
print('NBC  ->', all_clean[all_clean['label']==0]['headline'].sample(8, random_state=7).tolist())

=== Same-day headlines on overlapping topics ===

FOX  -> ["Michael Wilbon reveals Jordan's true feelings about LeBron", "Trump's patience with Putin 'runs out' as White House readies major trade punishment and more top headlines", "'Wheel of Fortune' contestant makes show history with record-breaking million-dollar win", 'How to save any file as a PDF', 'We are pulling the plug on med school DEI and making us all healthier as a result', 'Illegal immigrant truck driver accused of killing Indiana man after running red light', 'Housing costs are crushing families – here’s the way out', "Whitesnake frontman David Coverdale says it’s time to ‘hang up my rock n' roll platform shoes’ after 50 years"]

NBC  -> ['SAG-AFTRA reaches tentative deal with major studios', 'Israel launches airstrikes in Gaza after deadly West Bank raid sparked rocket fire', 'Chuck Schumer appeals to Harris to attend Al Smith dinner at request of prominent Cardinal', 'Bondi announces two more arrests in Minnesota chur

## 3. Adversarial validation: quantifying temporal drift

How different is our temporal validation set from the rest of the data? Train a classifier to *distinguish train from temp-val*. AUC near 0.5 = no drift; AUC near 1.0 = severe drift.

This is what motivated training on `co_val_temp` (course + val + temp) for the final submission — we needed every example we could get from the test distribution.

In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

# Build adversarial dataset: train (label=0) vs temp_val (label=1)
train_pool = course_only[['headline']].copy()
train_pool['adv_label'] = 0
temp_pool = temp_val[['headline']].copy()
temp_pool['adv_label'] = 1
adv = pd.concat([train_pool, temp_pool]).sample(frac=1, random_state=7).reset_index(drop=True)

vec = TfidfVectorizer(ngram_range=(1,2), min_df=2, max_features=20000)
X = vec.fit_transform(adv['headline'])
y = adv['adv_label'].values

auc = cross_val_score(LogisticRegression(max_iter=300, C=1.0), X, y, scoring='roc_auc', cv=5).mean()
print(f'Adversarial AUC (train vs temp-val): {auc:.3f}')
print('  0.5 = no drift; >0.85 = severe drift')
print()

# Top discriminative tokens for each side
lr = LogisticRegression(max_iter=300, C=1.0).fit(X, y)
feat = np.array(vec.get_feature_names_out())
coef = lr.coef_[0]
print('Top tokens distinctive of TRAIN (older):    ', feat[coef.argsort()[:8]].tolist())
print('Top tokens distinctive of TEMP-VAL (newer): ', feat[coef.argsort()[-8:][::-1]].tolist())

Adversarial AUC (train vs temp-val): 0.885
  0.5 = no drift; >0.85 = severe drift

Top tokens distinctive of TRAIN (older):     ['harris', 'biden', 'israel', 'hamas', 'gaza', 'jan', 'israeli', 'election']
Top tokens distinctive of TEMP-VAL (newer):  ['iran', 'ice', 'iran war', 'news', 'after', 'desk', '2026', 'cup']


**Interpretation:** AUC ≈ 0.69 = mild-to-moderate drift. The discriminative tokens are *named entities* (`gaza`, `harris`, `biden` vs `kimmel`, `derby`, `mayor`). A model that latches onto entities will degrade. **A model that learns *style* will not.**

## 4. Baseline: TF-IDF + Logistic Regression

The course baseline. Anchors the rest of the work.

In [7]:
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score

baseline = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1,2), min_df=2, max_features=20000)),
    ('lr',    LogisticRegression(max_iter=400, C=2.0))
])
baseline.fit(course_only['headline'], course_only['label'])

pred_val  = baseline.predict(val_frozen['headline'])
pred_temp = baseline.predict(temp_val['headline'])
print(f'Baseline TF-IDF + LR on val_frozen      : {accuracy_score(val_frozen["label"], pred_val):.4f}')
print(f'Baseline TF-IDF + LR on temporal_val    : {accuracy_score(temp_val["label"], pred_temp):.4f}')
print(f'Course-released baseline on hidden test : 0.6649  (ceiling without further effort)')

Baseline TF-IDF + LR on val_frozen      : 0.6904
Baseline TF-IDF + LR on temporal_val    : 0.6480
Course-released baseline on hidden test : 0.6649  (ceiling without further effort)


## 5. The stylometric pipeline — "style beats topic"

We hand-engineered **18 style features** that capture *how* headlines are written, independent of *what* they're about: punctuation density, capitalization patterns, quote/colon/dash usage, mean word length, etc.

In [8]:
import string

STOPWORDS = set('a an the and or but if then is are was were be been have has had do does did will would could should may to of in on at by for with about'.split())

def stylo_features(texts):
    out = np.zeros((len(texts), 18), dtype=np.float32)
    for i, t in enumerate(texts):
        if not isinstance(t, str): t = str(t)
        n = max(len(t), 1); words = t.split(); nw = max(len(words), 1)
        n_alpha = sum(1 for c in t if c.isalpha())
        n_upper = sum(1 for c in t if c.isupper())
        n_digit = sum(1 for c in t if c.isdigit())
        n_punct = sum(1 for c in t if c in string.punctuation)
        n_q = t.count('?'); n_excl = t.count('!')
        n_quote = t.count(chr(39)) + t.count(chr(34))
        n_colon = t.count(':'); n_dash = t.count('-'); n_comma = t.count(',')
        n_caps_words = sum(1 for w in words if w.isupper() and len(w) >= 2)
        n_titlecase  = sum(1 for w in words if w[:1].isupper() and not w.isupper())
        n_stop = sum(1 for w in words if w.lower() in STOPWORDS)
        avg_wl = float(np.mean([len(w) for w in words])) if words else 0.0
        n_space = sum(1 for c in t if c.isspace())
        out[i] = [n, nw, n_upper/max(n_alpha,1), n_digit/n, n_punct/n,
                  float(n_q>0), float(n_excl>0), n_q, n_excl, n_quote,
                  n_colon, n_dash, n_comma, n_caps_words/nw, n_titlecase/nw,
                  n_stop/nw, avg_wl, n_space/n]
    return out

feature_names = ['len_chars','len_words','upper_ratio','digit_ratio','punct_ratio',
                 'has_q','has_excl','n_q','n_excl','n_quote','n_colon','n_dash',
                 'n_comma','caps_word_frac','titlecase_frac','stop_frac','avg_wl','space_ratio']

fox_feats = stylo_features(course_only[course_only['label']==1]['headline'].tolist())
nbc_feats = stylo_features(course_only[course_only['label']==0]['headline'].tolist())

diffs = pd.DataFrame({
    'feature': feature_names,
    'fox_mean': fox_feats.mean(axis=0),
    'nbc_mean': nbc_feats.mean(axis=0),
})
diffs['delta_pct'] = 100 * (diffs['fox_mean'] - diffs['nbc_mean']) / (diffs['nbc_mean'].abs() + 1e-6)
diffs.reindex(diffs['delta_pct'].abs().sort_values(ascending=False).index).head(8)

,feature,fox_mean,nbc_mean,delta_pct
10,n_colon,0.327000,0.086063,279.949646
9,n_quote,1.220000,0.468073,160.642624
8,n_excl,0.004000,0.001666,140.049271
6,has_excl,0.004000,0.001666,140.049271
12,n_comma,0.461500,0.282066,63.614239
7,n_q,0.028500,0.051638,-44.807198
4,punct_ratio,0.023447,0.016372,43.213989
5,has_q,0.028500,0.048306,-41.000874


Notice: **caps-word fraction** and **uppercase ratio** are dramatically higher in Fox. **Stopword fraction** is higher in NBC. These are *style* signals invisible to topic-only models.

Now train: TF-IDF (90k features) **+** stylo (18 features), fed to LR.

In [ ]:
from sklearn.preprocessing import StandardScaler
from scipy.sparse import csr_matrix, hstack as sp_hstack

# TF-IDF char-ngrams + word-ngrams
tfidf_char = TfidfVectorizer(analyzer='char_wb', ngram_range=(2,4), min_df=3, max_features=60000)
tfidf_word = TfidfVectorizer(analyzer='word', ngram_range=(1,2), min_df=2, max_features=30000)
scaler = StandardScaler(with_mean=False)

Xc = tfidf_char.fit_transform(course_only['headline'])
Xw = tfidf_word.fit_transform(course_only['headline'])
Xs = scaler.fit_transform(stylo_features(course_only['headline'].tolist()))
Xtr = sp_hstack([Xc, Xw, csr_matrix(Xs)]).tocsr()

stylo_lr = LogisticRegression(max_iter=400, C=2.0).fit(Xtr, course_only['label'])

def stylo_predict(texts):
    Xc_ = tfidf_char.transform(texts)
    Xw_ = tfidf_word.transform(texts)
    Xs_ = scaler.transform(stylo_features(list(texts)))
    return stylo_lr.predict(sp_hstack([Xc_, Xw_, csr_matrix(Xs_)]).tocsr())

print(f'Stylo pipeline on val_frozen     : {accuracy_score(val_frozen["label"], stylo_predict(val_frozen["headline"])):.4f}')
print(f'Stylo pipeline on temporal_val   : {accuracy_score(temp_val["label"], stylo_predict(temp_val["headline"])):.4f}')
print(f'  ↑ stylo alone is the strongest *individual* held-out member of our ensemble')

## 6. Architectural diversity — the central insight

Our final ensemble has 5 members with deliberately **different inductive biases**:

| # | Model | Inductive bias |
|---|---|---|
| 1 | DistilBERT-base | distilled bidirectional, lexical patterns |
| 2 | ALBERT-base-v2  | parameter-shared layers |
| 3 | MPNet-base      | masked + permuted pretraining |
| 4 | ConvBERT-base   | mixed convolution + attention |
| 5 | Stylo (TF-IDF + 18 hand features) | topic-independent style |

Why does this work? Because their **errors are decorrelated**. Below is the actual correctness-correlation matrix from our held-out evaluation:

In [ ]:
# Real numbers from our held-out experiments (training each on course_only, evaluating on val+temp)
members = ['DistilBERT', 'ALBERT', 'MPNet', 'ConvBERT', 'Stylo']
corr = np.array([
    [1.000, 0.517, 0.517, 0.543, 0.192],
    [0.517, 1.000, 0.468, 0.497, 0.180],
    [0.517, 0.468, 1.000, 0.595, 0.278],
    [0.543, 0.497, 0.595, 1.000, 0.260],
    [0.192, 0.180, 0.278, 0.260, 1.000],
])

fig, ax = plt.subplots(figsize=(7, 5.5))
sns.heatmap(corr, annot=True, fmt='.3f', xticklabels=members, yticklabels=members,
            cmap='RdYlGn_r', vmin=0, vmax=1, ax=ax, cbar_kws={'label': 'Pairwise correctness correlation'})
ax.set_title('Held-out pairwise correctness correlation\n(higher = redundant errors; lower = complementary signal)')
plt.tight_layout(); plt.show()

print('\nKey observation:')
print(f'  Transformer pairs correlate at 0.47 - 0.60')
print(f'  Stylo correlates at only 0.18 - 0.28')
print(f'  → Stylo brings new information that transformers cannot capture.')

## 7. Saturation: when ensembling stops helping

We trained five more architectures (DeBERTa-v1, CANINE, XLNet, GPT-2, T5) and tried adding each to the 5-member ensemble. **Every single one regressed.** This is the saturation we observed:

In [ ]:
# Real leaderboard scores from our 215+ submission iteration
saturation = pd.DataFrame([
    ('5 (final)',                                       85.75, 'sub_144'),
    ('6 (+DeBERTa-v1)',                                 84.58, 'sub_188'),
    ('6 (+CANINE)',                                     83.50, 'sub_192'),
    ('7 (+DeBERTa+CANINE)',                             83.42, 'sub_191'),
    ('8 (+DeBERTa+CANINE+ConvBERT-s11)',                83.08, 'sub_193'),
], columns=['Ensemble size', 'Leaderboard %', 'Submission'])

fig, ax = plt.subplots(figsize=(9, 4.5))
colors = ['#2ca02c'] + ['#d62728']*4
ax.bar(range(len(saturation)), saturation['Leaderboard %'], color=colors, edgecolor='black')
for i, (size, score, sub) in enumerate(saturation.values):
    ax.text(i, score + 0.1, f'{score:.2f}%', ha='center', fontsize=10, fontweight='bold')
    ax.text(i, 80.5, sub, ha='center', fontsize=8, alpha=0.6)
ax.set_xticks(range(len(saturation)))
ax.set_xticklabels(saturation['Ensemble size'], rotation=15, ha='right')
ax.set_ylim(82, 87)
ax.set_ylabel('Hidden-test accuracy (%)')
ax.set_title('Adding more transformers monotonically regresses\n(5-member final beats every 6/7/8-member variant)')
ax.axhline(85.75, ls='--', color='gray', alpha=0.5)
ax.text(4, 85.85, 'sub_144 ceiling', fontsize=9, color='gray')
plt.tight_layout(); plt.show()

saturation

**The intuition:** with weight $1/n$ on each model, a 6th transformer dilutes the strong members. The optimal recipe is *not* the largest ensemble — it's the most *complementary* one.

## 8. Score progression — the full journey

From baseline to final, every gain came from a specific intervention:

In [ ]:
progression = pd.DataFrame([
    ('TF-IDF + LR baseline',                                      66.49),
    ('DistilBERT, full 48k scrape',                               76.83),
    ('DistilBERT on (course + temp_val) — distribution-matched',  80.58),
    ('+ best_headline FeatPlusStylo ensemble',                    82.25),
    ('2-seed DistilBERT + stylo',                                 83.25),
    ('+ ALBERT + MPNet (architectural diversity)',                84.75),
    ('+ ModernBERT (5 archs)',                                    85.42),
    ('swap ModernBERT → ConvBERT (final, sub_144)',               85.75),
], columns=['Recipe', 'Score'])

fig, ax = plt.subplots(figsize=(11, 5.5))
ax.plot(range(len(progression)), progression['Score'], marker='o', linewidth=2.5, markersize=11, color='#1f77b4')
ax.fill_between(range(len(progression)), progression['Score'], 65, alpha=0.15, color='#1f77b4')
for i, (recipe, score) in enumerate(progression.values):
    ax.text(i, score + 0.5, f'{score:.2f}%', ha='center', fontsize=10, fontweight='bold')
ax.set_xticks(range(len(progression)))
ax.set_xticklabels(progression['Recipe'], rotation=18, ha='right', fontsize=9)
ax.set_ylim(64, 88)
ax.axhline(66.49, ls=':', color='gray', alpha=0.5); ax.text(0, 67, 'baseline', fontsize=9, color='gray')
ax.axhline(85.75, ls=':', color='green', alpha=0.5); ax.text(0, 86.4, 'final 85.75%', fontsize=9, color='green')
ax.set_ylabel('Hidden-test accuracy (%)')
ax.set_title('Score progression: 66.49% → 85.75% (+19.26 pp over baseline)')
plt.tight_layout(); plt.show()

## 9. Take-aways

1. **Distribution-matched data > more data.** Training on 7.5k matched examples beat training on 48k by ~4 pp.
2. **Architectural diversity > seed diversity.** Four different transformer biases + stylo > seven DistilBERT seeds.
3. **Ensemble accuracy saturates.** The 6th transformer hurt; the 7th hurt more. 5 was the sweet spot.
4. **Style beats topic.** A hand-engineered stylometric pipeline was the **single strongest member** held-out (0.819 vs. 0.730–0.777 for the transformers), and beat zero-shot Claude Opus / GPT-5 (capped at 80%).
5. **Final result: 85.75%** — `+19.26 pp` over baseline, `+5.33 pp` over the next visible team.

---
